# Publicando uma tabela no catálogo

## Task 4: publicar uma tabela que o time consiga consultar pelo nome

## O problema

As views somem quando a sessão termina. A task: **publicar uma tabela** no **catálogo**, o lugar onde ficam os nomes
das tabelas e os seus **metadados** (colunas, tipos e onde está o dado).

In [1]:
import sys
sys.path.insert(0, "../../src")

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("arena-catalogo")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .getOrCreate()
)

RAW = "../../data/raw"

spark

A célula abaixo já vem pronta: o código dos vídeos 2.4 e 3.1, na ordem, com as cinco views do projeto.

In [2]:
# vídeo 2.4: escalações e Copas
from wc_pipeline.schemas import PLAYERS, WORLD_CUPS

escalacoes = (
    spark.read.option("header", "true").schema(PLAYERS).csv(f"{RAW}/WorldCupPlayers.csv")
    .select(
        F.col("MatchID").alias("match_id"),
        F.col("Team Initials").alias("sigla"),
        F.col("Player Name").alias("jogador"),
        F.col("Position").alias("posicao"),
    )
)

copas = (
    spark.read.option("header", "true").schema(WORLD_CUPS).csv(f"{RAW}/WorldCups.csv")
    .select(
        F.col("Year").alias("ano"),
        F.col("Country").alias("sede"),
        F.col("Winner").alias("campeao"),
        F.col("MatchesPlayed").alias("partidas_oficiais"),
        F.col("GoalsScored").alias("gols_oficiais"),
    )
)

# vídeo 3.1: as tabelas do projeto
from wc_pipeline.schemas import MATCHES
from wc_pipeline.transformacoes import limpar_partidas, partidas_por_selecao, resumo_por_selecao

partidas = limpar_partidas(
    spark.read.option("header", "true").schema(MATCHES).csv(f"{RAW}/WorldCupMatches.csv")
)
por_selecao = partidas_por_selecao(partidas)
resumo = resumo_por_selecao(por_selecao)

# vídeo 3.1: as views
partidas.createOrReplaceTempView("partidas")
por_selecao.createOrReplaceTempView("por_selecao")
resumo.createOrReplaceTempView("resumo")
escalacoes.createOrReplaceTempView("escalacoes")
copas.createOrReplaceTempView("copas")

A próxima célula também vem pronta: ela só apaga as pastas da demonstração (`data/warehouse/copa.db` e
`data/externas`), para o notebook poder rodar de novo.

In [3]:
import shutil
for pasta in ["../../data/warehouse/copa.db", "../../data/externas"]:
    shutil.rmtree(pasta, ignore_errors=True)

## View ou tabela?

| Objeto | Como cria | Grava dado? | Vive enquanto... |
|---|---|---|---|
| temp view | `createOrReplaceTempView` | não | a `SparkSession` existir |
| tabela | `saveAsTable` ou `CREATE TABLE` | **sim** | alguém apagar (com uma ressalva) |

In [6]:
spark.sql("SHOW TABLES").show()

+---------+-----------+-----------+
|namespace|  tableName|isTemporary|
+---------+-----------+-----------+
|         |      copas|       true|
|         | escalacoes|       true|
|         |   partidas|       true|
|         |por_selecao|       true|
|         |     resumo|       true|
+---------+-----------+-----------+



## Publicando uma tabela: `saveAsTable`

In [7]:
spark.sql("CREATE DATABASE IF NOT EXISTS copa")

resumo.write.mode("overwrite").saveAsTable("copa.resumo_selecoes")

spark.sql("SELECT ano, selecao, gols_pro, saldo FROM copa.resumo_selecoes ORDER BY saldo DESC LIMIT 5").show()

+----+-------+--------+-----+
| ano|selecao|gols_pro|saldo|
+----+-------+--------+-----+
|1954|Hungary|      27|   17|
|1950| Brazil|      22|   16|
|2014|Germany|      18|   14|
|2002| Brazil|      18|   14|
|1998| France|      15|   13|
+----+-------+--------+-----+



In [10]:
(
    spark.sql("DESCRIBE EXTENDED copa.resumo_selecoes")
    .show(truncate=False)
)

+----------------------------+-------------------------------------------------------------+-------+
|col_name                    |data_type                                                    |comment|
+----------------------------+-------------------------------------------------------------+-------+
|ano                         |int                                                          |NULL   |
|selecao                     |string                                                       |NULL   |
|jogos                       |bigint                                                       |NULL   |
|vitorias                    |bigint                                                       |NULL   |
|empates                     |bigint                                                       |NULL   |
|derrotas                    |bigint                                                       |NULL   |
|gols_pro                    |bigint                                                       

- **Type**: `MANAGED` (gerenciada: o Spark é dono dos arquivos) ou `EXTERNAL`.
- **Provider**: o formato dos arquivos (Parquet é o tema da aula 5).
- **Location**: a pasta dos arquivos, dentro do `spark.sql.warehouse.dir` da sessão. Abra essa pasta no explorador de
  arquivos e veja os arquivos da tabela.

## Gerenciada ou externa?

Tabela **externa**: criada com `LOCATION`, numa pasta escolhida por você. No `DROP TABLE`:

- **gerenciada**: apaga o registro **e os arquivos**;
- **externa**: apaga só o registro; os arquivos ficam.

`LOCATION` recebe o caminho completo da pasta; `Path(...).resolve()` monta esse caminho.

In [11]:
from pathlib import Path

pasta_externa = Path("../../data/externas/copa_resumo").resolve()

spark.sql(f"""
    CREATE TABLE copa.resumo_externa
    USING PARQUET
    LOCATION '{pasta_externa}'
    AS SELECT * FROM copa.resumo_selecoes
""")

spark.catalog.listTables("copa")

[Table(name='resumo_externa', catalog='spark_catalog', namespace=['copa'], description=None, tableType='EXTERNAL', isTemporary=False),
 Table(name='resumo_selecoes', catalog='spark_catalog', namespace=['copa'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='copas', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='escalacoes', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='partidas', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='por_selecao', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='resumo', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [12]:
spark.sql("DROP TABLE copa.resumo_externa")

print("registro existe?", spark.catalog.tableExists("copa.resumo_externa"))
print("arquivos existem?", any(pasta_externa.glob("*.parquet")))

registro existe? False
arquivos existem? True


## O que sobrevive quando a sessão termina?

Simulando o "dia seguinte": encerrar a sessão e abrir outra.

In [13]:
spark.stop()

spark = (
    SparkSession.builder
    .appName("arena-catalogo-dia-seguinte")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .getOrCreate()
)

pasta_da_tabela = Path("../../data/warehouse/copa.db/resumo_selecoes").resolve()

print("bancos:", [banco.name for banco in spark.catalog.listDatabases()])
print("tabelas:", [tabela.name for tabela in spark.catalog.listTables()])
print("arquivos da tabela ainda no disco?", any(pasta_da_tabela.glob("*.parquet")))

spark

bancos: ['default']
tabelas: []
arquivos da tabela ainda no disco? True


In [14]:
spark.sql("CREATE DATABASE IF NOT EXISTS copa")
spark.sql(f"""
    CREATE TABLE copa.resumo_selecoes
    USING PARQUET
    LOCATION '{pasta_da_tabela}'
""")

spark.sql("SELECT COUNT(*) AS linhas FROM copa.resumo_selecoes").show()

+------+
|linhas|
+------+
|   489|
+------+



Registrar a tabela de novo com `LOCATION` recupera o dado a partir dos arquivos. Agora ela é **externa**.

Em produção, um catálogo **persistente** (Hive Metastore, AWS Glue Data Catalog, Unity Catalog) guarda os metadados
fora da sessão.

## Demanda da aula 3 atendida

| Etapa | Entrega |
|---|---|
| Task 1 | as cinco views do projeto em SQL |
| Task 2 | estatísticas em SQL idênticas às da aula 2, e `sql/gols_por_copa.sql` |
| Task 3 | a seleção que mais marcou em cada Copa |
| Task 4 | `copa.resumo_selecoes` publicada, e o que sobrevive à sessão |

## Para consultar depois

- **Global temp view** (`createOrReplaceGlobalTempView`): uma view que vale para todas as sessões da mesma aplicação,
  no banco `global_temp` (`SELECT * FROM global_temp.nome`). Some quando a aplicação termina.
- **`spark.catalog`:** `listDatabases()`, `listTables("banco")`, `listColumns("tabela")` e `tableExists("banco.tabela")`
  mostram os metadados como objetos Python.
- **`spark.table("copa.resumo_selecoes")`** lê uma tabela do catálogo como DataFrame.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [15]:
spark.stop()